In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Impor fungsi ekstraksi dari file feature_extraction.py
from feature_extraction import extract_lbp, extract_hog, extract_glcm

print("Menyiapkan simulasi dataset tekstur kustom...")

# Membuat 150 gambar tekstur tiruan (3 kelas tekstur berbeda: Garis Horizontal, Vertikal, Ring/Bulat)
np.random.seed(42)
X_images = []
y_labels = []

for i in range(50):
    # Kelas 0: Tekstur Garis Horizontal
    img = np.zeros((64, 64), dtype=np.uint8)
    img[::4, :] = 255
    img += np.random.randint(0, 50, (64, 64), dtype=np.uint8) # Tambah noise
    X_images.append(img)
    y_labels.append(0)
    
    # Kelas 1: Tekstur Garis Vertikal
    img = np.zeros((64, 64), dtype=np.uint8)
    img[:, ::4] = 255
    img += np.random.randint(0, 50, (64, 64), dtype=np.uint8)
    X_images.append(img)
    y_labels.append(1)

    # Kelas 2: Tekstur Kotak/Catur
    img = np.zeros((64, 64), dtype=np.uint8)
    img[::4, ::4] = 255
    img += np.random.randint(0, 50, (64, 64), dtype=np.uint8)
    X_images.append(img)
    y_labels.append(2)

X_images = np.array(X_images)
y_labels = np.array(y_labels)

print(f"Sukses membuat {len(X_images)} sampel gambar dengan 3 kelas tekstur berbeda!")

In [ ]:
print("Memulai proses ekstraksi fitur tekstur (LBP, HOG, GLCM)...")

X_lbp = []
X_hog = []
X_glcm = []

for img in X_images:
    # Ekstrak masing-masing fitur dari setiap gambar
    X_lbp.append(extract_lbp(img))
    X_hog.append(extract_hog(img))
    X_glcm.append(extract_glcm(img))

# Ubah menjadi numpy array agar siap dimasukkan ke classifier
X_lbp = np.array(X_lbp)
X_hog = np.array(X_hog)
X_glcm = np.array(X_glcm)

print(f"Selesai! Dimensi matriks fitur baru:")
print(f"- Fitur LBP : {X_lbp.shape}")
print(f"- Fitur HOG : {X_hog.shape}")
print(f"- Fitur GLCM: {X_glcm.shape}")

In [ ]:
# Dictionary untuk menampung hasil akurasi dari 6 kombinasi
hasil_akurasi = {
    'LBP': {},
    'HOG': {},
    'GLCM': {}
}

# Gabungkan fitur dan tipenya ke dalam dictionary untuk looping otomatis
fitur_dict = {'LBP': X_lbp, 'HOG': X_hog, 'GLCM': X_glcm}

# Definisikan 2 jenis classifier sesuai instruksi tugas
classifiers = {
    'SVM': SVC(kernel='linear', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Looping untuk menguji 6 kombinasi berbeda
for nama_fitur, data_fitur in fitur_dict.items():
    # Split data menjadi Train 80% dan Test 20%
    X_train, X_test, y_train, y_test = train_test_split(data_fitur, y_labels, test_size=0.2, random_state=42)
    
    for nama_clf, clf in classifiers.items():
        # Latih model classifier
        clf.fit(X_train, y_train)
        # Lakukan prediksi
        y_pred = clf.predict(X_test)
        # Hitung akurasi
        acc = accuracy_score(y_test, y_pred)
        
        # Simpan hasil akurasi
        hasil_akurasi[nama_fitur][nama_clf] = acc
        print(f"Kombinasi [{nama_fitur} + {nama_clf:13s}] -> Akurasi: {acc * 100:.2f}%")

# --- MEMBUAT VISUALISASI GRAFIK BAR CHART ---
fitur_labels = list(hasil_akurasi.keys())
akurasi_svm = [hasil_akurasi[f]['SVM'] * 100 for f in fitur_labels]
akurasi_rf = [hasil_akurasi[f]['Random Forest'] * 100 for f in fitur_labels]

x = np.arange(len(fitur_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bar1 = ax.bar(x - width/2, akurasi_svm, width, label='SVM', color='#4f46e5')
bar2 = ax.bar(x + width/2, akurasi_rf, width, label='Random Forest', color='#f59e0b')

ax.set_ylabel('Rata-rata Akurasi (%)')
ax.set_title('Perbandingan Performa Kombinasi Ekstraksi Fitur & Classifier Tekstur')
ax.set_xticks(x)
ax.set_xticklabels(fitur_labels)
ax.set_ylim(0, 110)
ax.legend()

# Fungsi untuk memunculkan angka persentase di atas batang grafik
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  
                    textcoords="offset points",
                    ha='center', va='bottom')

autolabel(bar1)
autolabel(bar2)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()